# Partie I — MLP et données tabulaires

**Projet Deep Learning — EMSI Casablanca (2025-2026)**

Notebook **autonome** : tout le code de la Partie I est ici (aucun import depuis un module externe du projet).

**Objectif** — Classifier des tumeurs du sein (bénignes vs malignes) avec le dataset *Breast Cancer Wisconsin*.

**Plan**
1. Configuration et utilitaires
2. Chargement, EDA et prétraitement
3. Architectures MLP (`Sequential` vs `Module` personnalisé)
4. Entraînement avec early stopping
5. Comparaison des initialisations de poids
6. Export des figures/tableaux pour le rapport LaTeX (`../results/`)

In [ ]:
# Dépendances : pip install torch scikit-learn matplotlib seaborn pandas numpy tqdm

import json
import os
import random
from copy import deepcopy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

PROJECT_ROOT = Path("..").resolve()
FIGURES_DIR = PROJECT_ROOT / "results" / "figures" / "mlp"
TABLES_DIR = PROJECT_ROOT / "results" / "tables" / "mlp"
MODELS_DIR = PROJECT_ROOT / "results" / "saved_models" / "mlp"
LOGS_DIR = PROJECT_ROOT / "results" / "logs" / "mlp"
for folder in [FIGURES_DIR, TABLES_DIR, MODELS_DIR, LOGS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

SEED = 42
CONFIG = {
    "test_size": 0.15,
    "val_size": 0.15,
    "batch_size": 32,
    "num_workers": 0,
    "hidden_dims": [64, 32],
    "dropout": 0.15,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "epochs": 30,
    "patience": 6,
    "gaussian_std": 0.02,
    "constant_value": 0.05,
    "initializations": ["gaussian", "constant", "xavier"],
}


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


set_seed(SEED)
device = get_device()
print(f"Périphérique : {device}")
print(f"Racine projet : {PROJECT_ROOT}")

## 1. Utilitaires — métriques et visualisation

In [ ]:
def classification_metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="weighted", zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, average="weighted", zero_division=0)),
        "f1_score": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "confusion_matrix": confusion_matrix(y_true, y_pred),
    }


def save_dataframe(df, csv_path, latex_path=None):
    df.to_csv(csv_path, index=False)
    if latex_path is not None:
        latex_path.write_text(
            df.to_latex(index=False, float_format="%.4f", escape=True),
            encoding="utf-8",
        )


def plot_training_curves(history, title=""):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history["train_loss"], label="Train")
    axes[0].plot(history["val_loss"], label="Val")
    axes[0].set_title(f"{title} — pertes")
    axes[0].set_xlabel("Époque")
    axes[0].set_ylabel("Perte")
    axes[0].legend()
    axes[1].plot(history["train_accuracy"], label="Train acc")
    axes[1].plot(history["val_accuracy"], label="Val acc")
    axes[1].set_title(f"{title} — accuracy")
    axes[1].set_xlabel("Époque")
    axes[1].legend()
    plt.tight_layout()
    return fig


def plot_confusion_matrix(matrix, labels, title=""):
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        matrix,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=labels,
        yticklabels=labels,
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel("Prédiction")
    ax.set_ylabel("Vérité terrain")
    plt.tight_layout()
    return fig

## 2. Chargement et exploration des données

In [ ]:
dataset = load_breast_cancer(as_frame=True)
df = dataset.data.copy()
df["target"] = dataset.target
df = df.drop_duplicates().dropna()
class_names = list(dataset.target_names)

print(f"Échantillons : {len(df)} | Features : {df.shape[1] - 1}")
print(f"Classes : {class_names}")
display(df.head())

In [ ]:
sns.set_theme(style="whitegrid", context="talk")

counts = df["target"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar([class_names[i] for i in counts.index], counts.values, color=["#d62728", "#2ca02c"])
ax.set_title("Distribution des classes — Breast Cancer Wisconsin")
ax.set_ylabel("Nombre d'échantillons")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "mlp_class_distribution.png", dpi=200, bbox_inches="tight")
plt.show()

top_features = dataset.data.iloc[:, :10]
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(top_features.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Corrélation — 10 premières variables")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "mlp_correlation_heatmap.png", dpi=200, bbox_inches="tight")
plt.show()

key_features = ["mean radius", "mean texture", "mean perimeter", "mean area", "mean smoothness"]
plot_df = df.copy()
plot_df["diagnosis"] = plot_df["target"].map({0: class_names[0], 1: class_names[1]})
fig, axes = plt.subplots(1, len(key_features), figsize=(16, 4))
for ax, feat in zip(axes, key_features):
    sns.boxplot(data=plot_df, x="diagnosis", y=feat, ax=ax)
    ax.set_title(feat)
    ax.tick_params(axis="x", rotation=15)
plt.suptitle("Distribution de variables clés par classe diagnostic")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "mlp_feature_boxplots.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
X = df.drop(columns=["target"])
y = df["target"]

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=CONFIG["test_size"], random_state=SEED, stratify=y
)
val_ratio = CONFIG["val_size"] / (1.0 - CONFIG["test_size"])
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=val_ratio, random_state=SEED, stratify=y_train_val
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)


def make_loader(X_scaled, y_series, shuffle):
    tensor_x = torch.tensor(X_scaled, dtype=torch.float32)
    tensor_y = torch.tensor(y_series.to_numpy(), dtype=torch.long)
    dataset_split = TensorDataset(tensor_x, tensor_y)
    return DataLoader(
        dataset_split,
        batch_size=CONFIG["batch_size"],
        shuffle=shuffle,
        num_workers=CONFIG["num_workers"],
    )


train_loader = make_loader(X_train_s, y_train, shuffle=True)
val_loader = make_loader(X_val_s, y_val, shuffle=False)
test_loader = make_loader(X_test_s, y_test, shuffle=False)
input_dim = X.shape[1]
num_classes = len(class_names)

summary = pd.DataFrame(
    [
        {"split": "entraînement", "n_samples": len(y_train), "n_features": input_dim},
        {"split": "validation", "n_samples": len(y_val), "n_features": input_dim},
        {"split": "test", "n_samples": len(y_test), "n_features": input_dim},
        {"split": "total", "n_samples": len(df), "n_features": input_dim},
    ]
)
save_dataframe(
    summary,
    TABLES_DIR / "mlp_dataset_summary.csv",
    TABLES_DIR / "mlp_dataset_summary.tex",
)
display(summary)

## 3. Architectures MLP

In [ ]:
def build_sequential_mlp(input_dim, hidden_dims, num_classes, dropout):
    layers = []
    previous_dim = input_dim
    for hidden_dim in hidden_dims:
        layers.extend([nn.Linear(previous_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout)])
        previous_dim = hidden_dim
    layers.append(nn.Linear(previous_dim, num_classes))
    return nn.Sequential(*layers)


class CustomMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, num_classes, dropout):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dims[0])
        self.fc2 = nn.Linear(hidden_dims[0], hidden_dims[1])
        self.dropout = nn.Dropout(dropout)
        self.output_layer = nn.Linear(hidden_dims[1], num_classes)
        self.activation = nn.ReLU()

    def forward(self, x):
        x = self.dropout(self.activation(self.fc1(x)))
        x = self.dropout(self.activation(self.fc2(x)))
        return self.output_layer(x)


def initialize_weights(model, method, gaussian_std=0.02, constant_value=0.05):
    for module in model.modules():
        if isinstance(module, nn.Linear):
            if method == "gaussian":
                nn.init.normal_(module.weight, mean=0.0, std=gaussian_std)
            elif method == "constant":
                nn.init.constant_(module.weight, constant_value)
            elif method == "xavier":
                nn.init.xavier_uniform_(module.weight)
            nn.init.zeros_(module.bias)


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def inspect_model(model):
    lines = ["=== named_parameters() ==="]
    for name, param in model.named_parameters():
        lines.append(f"{name}: shape={tuple(param.shape)}, n={param.numel()}")
    lines.append(f"Paramètres entraînables : {count_params(model)}")
    return "\n".join(lines)


demo = build_sequential_mlp(input_dim, CONFIG["hidden_dims"], num_classes, CONFIG["dropout"])
print(inspect_model(demo))

## 4. Entraînement et évaluation

In [ ]:
def run_epoch(model, loader, criterion, optimizer, train=True):
    model.train(train)
    total_loss = 0.0
    correct = 0
    total = 0
    for inputs, targets in loader:
        inputs = inputs.to(device)
        targets = targets.to(device)
        if train:
            optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        if train:
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * inputs.size(0)
        correct += (outputs.argmax(dim=1) == targets).sum().item()
        total += inputs.size(0)
    return total_loss / total, correct / total


def train_classifier(model, train_loader, val_loader):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"]
    )
    history = {key: [] for key in ["train_loss", "val_loss", "train_accuracy", "val_accuracy"]}
    best_state = deepcopy(model.state_dict())
    best_val_loss = float("inf")
    best_epoch = 0
    patience_counter = 0

    for _ in range(CONFIG["epochs"]):
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, train=True)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer, train=False)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_accuracy"].append(train_acc)
        history["val_accuracy"].append(val_acc)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = len(history["val_loss"])
            best_state = deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
        if patience_counter >= CONFIG["patience"]:
            break

    model.load_state_dict(best_state)
    return model, history, best_epoch, best_val_loss


@torch.no_grad()
def evaluate_classifier(model, loader):
    model.eval()
    y_true, y_pred = [], []
    for inputs, targets in loader:
        predictions = model(inputs.to(device)).argmax(dim=1).cpu().numpy()
        y_pred.extend(predictions.tolist())
        y_true.extend(targets.numpy().tolist())
    return classification_metrics(y_true, y_pred)

## 5. Campagne expérimentale (2 architectures × 3 initialisations)

In [ ]:
def build_model(model_name):
    if model_name == "mlp_sequential":
        return build_sequential_mlp(input_dim, CONFIG["hidden_dims"], num_classes, CONFIG["dropout"])
    return CustomMLP(input_dim, CONFIG["hidden_dims"], num_classes, CONFIG["dropout"])


results = []
best_info = None

for model_name in ["mlp_sequential", "mlp_custom"]:
    for init_name in CONFIG["initializations"]:
        print(f"\n>>> {model_name} | initialisation={init_name}")
        model = build_model(model_name).to(device)
        initialize_weights(model, init_name, CONFIG["gaussian_std"], CONFIG["constant_value"])
        log_path = LOGS_DIR / f"{model_name}_{init_name}_inspection.txt"
        log_path.write_text(inspect_model(model), encoding="utf-8")

        model, history, best_epoch, best_val_loss = train_classifier(model, train_loader, val_loader)
        metrics = evaluate_classifier(model, test_loader)

        fig = plot_training_curves(history, f"{model_name} ({init_name})")
        fig.savefig(FIGURES_DIR / f"{model_name}_{init_name}_curves.png", dpi=200, bbox_inches="tight")
        plt.show()

        fig = plot_confusion_matrix(
            metrics["confusion_matrix"],
            class_names,
            f"Matrice de confusion — {model_name} ({init_name})",
        )
        fig.savefig(
            FIGURES_DIR / f"{model_name}_{init_name}_confusion_matrix.png",
            dpi=200,
            bbox_inches="tight",
        )
        plt.show()

        record = {
            "modele": model_name,
            "initialisation": init_name,
            "accuracy": metrics["accuracy"],
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "f1_score": metrics["f1_score"],
            "best_epoch": best_epoch,
            "best_val_loss": best_val_loss,
            "parametres_entrainables": count_params(model),
        }
        results.append(record)
        if best_info is None or record["f1_score"] > best_info["record"]["f1_score"]:
            best_info = {"record": record, "state_dict": model.state_dict()}

results_df = pd.DataFrame(results).sort_values("f1_score", ascending=False)
save_dataframe(
    results_df,
    TABLES_DIR / "mlp_comparaison_modeles.csv",
    TABLES_DIR / "mlp_comparaison_modeles.tex",
)
display(
    results_df.style.format(
        {
            "accuracy": "{:.4f}",
            "precision": "{:.4f}",
            "recall": "{:.4f}",
            "f1_score": "{:.4f}",
            "best_val_loss": "{:.4f}",
        }
    )
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
labels = [f"{row.modele}\n{row.initialisation}" for row in results_df.itertuples()]
ax.bar(labels, results_df["f1_score"], color="#1f77b4")
ax.set_title("Comparaison des scores F1 des MLP")
ax.set_ylabel("Score F1 pondéré")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "mlp_f1_comparison.png", dpi=200, bbox_inches="tight")
plt.show()

best_path = MODELS_DIR / "best_mlp_model.pt"
torch.save({"state_dict": best_info["state_dict"], "metadata": best_info["record"]}, best_path)
print("Meilleur modèle :", best_info["record"])

reloaded = build_model(best_info["record"]["modele"]).to(device)
checkpoint = torch.load(best_path, map_location=device)
reloaded.load_state_dict(checkpoint["state_dict"])
reload_metrics = evaluate_classifier(reloaded, test_loader)
print(
    "Vérification rechargement :",
    {key: reload_metrics[key] for key in ["accuracy", "precision", "recall", "f1_score"]},
)